In [ ]:
# ========== 导入：环境变量、OpenAI 客户端、Gradio、子进程与展示 ==========
# 导入

# 标准库 os：读环境变量（API Key）
import os
# 标准库 io：用 StringIO 捕获 exec / unittest 的标准输出
import io
# 标准库 sys：临时替换 sys.stdout，把打印重定向到 buffer
import sys
# 从 dotenv 导入 load_dotenv：把 .env 密钥读进进程环境
from dotenv import load_dotenv
# 从 openai 导入 OpenAI：Chat Completions 客户端（也可指向 OpenRouter）
from openai import OpenAI
# Gradio：搭建「原代码 / 测试代码」双栏 UI
import gradio as gr
# subprocess：备用导入（本格主要逻辑未必用到）
import subprocess
# IPython 展示工具：在笔记本里渲染 Markdown
from IPython.display import Markdown, display


In [ ]:
# ========== 加载 .env 并检查 OpenAI / OpenRouter Key 是否存在 ==========
# 加载 .env；override=True 允许覆盖已有环境变量
load_dotenv(override=True)
# 读取官方 OpenAI Key
openai_api_key = os.getenv('OPENAI_API_KEY')
# 读取 OpenRouter Key（本练习主要走聚合网关）
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

# 有 Key 时只打印前缀，避免把完整密钥打到日志里
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
# OpenRouter 可选；没有也能跑到后面，但调模型会失败

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


In [ ]:

# ========== 创建 OpenAI 默认客户端 + OpenRouter 兼容客户端 ==========
# 默认 OpenAI 客户端（读 OPENAI_API_KEY）
openai = OpenAI()

# OpenRouter 的 OpenAI 兼容 base_url（字符串勿改）
openrouter_url = "https://openrouter.ai/api/v1"

# 指向 OpenRouter：用同一套 chat.completions API 路由多家模型
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)


In [ ]:
# ========== 模型列表与 client 映射（全部走 OpenRouter） ==========
# 可选模型 id（字符串勿改；需与 OpenRouter 路由名一致）
models = ["openai/gpt-5.1-codex", "anthropic/claude-haiku-4.5", "gemini-2.5-flash", "openai/gpt-oss-120b"]

# 每个模型名 → 同一个 openrouter 客户端
clients = {
    "openai/gpt-5.1-codex": openrouter,
    "anthropic/claude-haiku-4.5": openrouter,
    "gemini-2.5-flash": openrouter,
    "openai/gpt-oss-120b": openrouter
}


In [ ]:


# ========== Prompt：让模型为用户代码写 unittest 测试 ==========
# system_prompt：测试专家人设 + 只输出代码（英文原文勿改）
system_prompt = f""" You are a python testing expert.
Your task is to create Python  test code for the python code provided by the user. The test code should be written in python and should be able to be run using unittest. The test code should cover all the functions in the provided python code and should include edge cases. The test code should be written in a way that it can be easily understood by other developers. The test code should be written in a way that it can be easily maintained and updated as the provided python code changes. The test code should be written in a way that it can be easily integrated into a CI/CD pipeline.
Respond only with  code. Do not provide any explanation other than occasional comments.
"""

# 把待测源码包进 fenced code block，作为 user 消息
def user_prompt_for(python):
    return f"""
Create test code for the following python code:
```python
{python}
```
"""


In [ ]:
# ========== 组装 messages：OpenAI 路由用 TypedDict，其它用普通 dict ==========
def messages_for(python, model):
    # OpenRouter 上 openai/* 路由：用官方类型化消息参数
    if model.startswith("openai/"):
        # 导入 Chat Completions 消息 TypedDict
        from openai.types.chat import ChatCompletionSystemMessageParam, ChatCompletionUserMessageParam
        return [
            ChatCompletionSystemMessageParam(role="system", content=system_prompt),
            ChatCompletionUserMessageParam(role="user", content=user_prompt_for(python))
        ]
    # 其它厂商：普通 dict 即可
    else:
        return [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt_for(python)}
        ]


In [ ]:
# ========== 待测样例：LCG 随机数 + 最大子数组和（给 UI 默认填入） ==========
# 这段 python_hard 是「被测代码」字符串，内容保持原文（含注释）
python_hard = """# Be careful to support large numbers

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value

def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# 参数
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# 计时功能
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""


In [ ]:
# ========== 调用选定模型：为 python 源码生成 unittest ==========
def port(model, python):
    # 按模型名取对应 client（这里都是 openrouter）
    client = clients[model]
    # 名称含 gpt 时打开高 reasoning_effort；其它模型传 None
    reasoning_effort = "high" if 'gpt' in model else None
    # Chat Completions：把 messages_for 组好的对话发给模型
    response = client.chat.completions.create(model=model, messages=messages_for(python, model), reasoning_effort=reasoning_effort)
    # 取出助手回复文本
    reply = response.choices[0].message.content
    # 去掉可能出现的代码围栏（含 cpp/rust 误标），只留纯代码
    reply = reply.replace('```cpp','').replace('```rust','').replace('```','')
    # 返回生成的测试代码字符串
    return reply


In [ ]:
# ========== 执行 Python / 若含 unittest 则自动跑测试套件 ==========
def run_python(code):
    # 延迟导入：动态类型检查与 unittest 运行器
    import types
    import unittest
    # 独立全局命名空间；保留完整 builtins（测试需要断言等）
    globals_dict = {"__builtins__": __builtins__}
    # 捕获 print / TextTestRunner 输出
    buffer = io.StringIO()
    # 保存原 stdout，finally 里恢复
    old_stdout = sys.stdout
    # 重定向标准输出到内存 buffer
    sys.stdout = buffer
    try:
        # 先 exec 用户/模型给出的代码（定义函数或 TestCase）
        exec(code, globals_dict)
        # 如果存在unittest，则运行测试
        # 源码提到 unittest 时，尝试收集并运行 TestCase
        if any('unittest' in line for line in code.splitlines()):
            # 查找 globals_dict 中的所有 TestCase 类
            # 查找 globals_dict 中的所有 TestCase 类
            test_cases = [obj for obj in globals_dict.values()
                          if isinstance(obj, type) and issubclass(obj, unittest.TestCase)]
            # 找到测试类则组装并运行套件
            if test_cases:
                # 拼装测试套件
                suite = unittest.TestSuite()
                for case in test_cases:
                    suite.addTests(unittest.defaultTestLoader.loadTestsFromTestCase(case))
                # verbosity=2：更详细的测试输出
                runner = unittest.TextTestRunner(stream=buffer, verbosity=2)
                # 执行全部测试用例
                runner.run(suite)
            else:
                # 源码提到 unittest 但没定义 TestCase
                buffer.write("No unittest.TestCase classes found in test code.\n")
        # 取出捕获到的全部输出
        output = buffer.getvalue()
    # exec 或收集测试失败：返回简短错误字符串
    except Exception as e:
        output = f"Error: {e}"
    # 无论成败都恢复 stdout，避免弄坏笔记本后续输出
    finally:
        sys.stdout = old_stdout

    # 把运行/测试输出返回给 Gradio
    return output


In [ ]:
# ========== Gradio UI：左原码 / 右测试码，一键生成并运行 ==========
# 从本地 styles 模块导入 CSS（需保证同目录有 styles.py）
from styles import CSS

# Monochrome 主题 + 自定义 CSS；title 保持原文
with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Create test Python") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            # 左侧：待测 Python（默认填入 python_hard）
            python = gr.Code(
                label="Python (original)",
                value=python_hard,
                language="python",
                lines=26
            )
        with gr.Column(scale=6):
            # 右侧：模型生成的测试代码（初始为空）
            python_test = gr.Code(
                label=f"Test code (generated)",
                value="",
                language="python",
                lines=26
            )

    with gr.Row(elem_classes=["controls"]):
        # 运行左侧原码
        python_run = gr.Button("Run Python", elem_classes=["run-btn", "py"])
        # 选择用于生成测试的模型
        model = gr.Dropdown(models, value=models[0], show_label=False)
        # 运行右侧测试代码
        python_run_test = gr.Button(f"Run test code", elem_classes=["convert-btn"])
        # 调用 port() 生成测试代码
        convert = gr.Button(f"Generate test code", elem_classes=["convert-btn"])

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            # 原码运行输出
            python_out = gr.TextArea(label="Python result", lines=8, elem_classes=["py-out"])
        with gr.Column(scale=6):
            # 测试运行输出
            python_test_out = gr.TextArea(label="Python test result", lines=8, elem_classes=["py-out"])
            # cpp_out = gr.TextArea(label=f"Python测试结果",lines=8, elem_classes=["py-out"])

    # 事件：生成测试
    convert.click(fn=port, inputs=[model, python], outputs=[python_test])
    # 事件：跑原码
    python_run.click(fn=run_python, inputs=[python], outputs=[python_out])
    # 事件：跑测试码
    python_run_test.click(fn=run_python, inputs=[python_test], outputs=[python_test_out])

# 启动界面并尝试打开浏览器
ui.launch(inbrowser=True)
